# 🏭 Notebook 1: Foundation — Synthetic Manufacturing Data & Unity Catalog Setup

## Genie Family PoC Workshop | Manufacturing Industry

**Purpose**: Create a realistic manufacturing operations dataset in Unity Catalog that powers all Genie product demos.

**What this notebook does**:
1. Creates a dedicated catalog (`genie_family_poc`) and schema (`manufacturing`)
2. Generates **10 production-quality synthetic tables** using `dbldatagen` (pre-installed on DBR 14.0+)
3. Adds **rich table and column comments** — the #1 driver of Genie accuracy
4. Creates **metric views** for governed metric definitions
5. Validates all assets are ready for Genie consumption

**Runtime**: ~5 minutes | **Cluster**: DBR 14.0+ | **No pip installs required**

---

### 📋 Prerequisites
- Databricks workspace with **Unity Catalog** enabled
- **CREATE CATALOG** or **USE CATALOG** permission
- **Serverless SQL Warehouse** provisioned
- **Partner-Powered AI** enabled (Account Console → Settings)

### 🏗️ What Gets Created

| # | Table | Description | Rows |
|---|---|---|---|
| 1 | `dim_plants` | Manufacturing plant locations | 20 |
| 2 | `dim_production_lines` | Production lines per plant | 80 |
| 3 | `dim_products` | Product catalog with specs | 200 |
| 4 | `dim_suppliers` | Raw material suppliers | 50 |
| 5 | `dim_employees` | Plant workforce | 500 |
| 6 | `fct_production_orders` | Production order transactions | 100,000 |
| 7 | `fct_quality_inspections` | Quality control records | 50,000 |
| 8 | `fct_equipment_telemetry` | IoT sensor readings | 200,000 |
| 9 | `fct_inventory_movements` | Raw material & finished goods | 80,000 |
| 10 | `agg_daily_plant_kpis` | Pre-aggregated daily KPIs | 7,300 |


## 🔧 Cell 1: Configuration Parameters

This cell defines all configurable parameters for the PoC. Change these values to customize the deployment for your customer's environment.

**Key parameters**:
- `CATALOG_NAME`: The Unity Catalog catalog to create (default: `genie_family_poc`)
- `SCHEMA_NAME`: The schema within the catalog (default: `manufacturing`)
- `NUM_ROWS_SCALE`: Multiplier for data volume (1 = default, 10 = 10x more data)

> 💡 **Tip**: For a quick demo, keep `NUM_ROWS_SCALE = 1`. For stress-testing Genie with larger datasets, increase to 5 or 10.


In [ ]:
# ============================================================
# CONFIGURATION — Modify these parameters for your environment
# ============================================================

CATALOG_NAME = "genie_family_poc"
SCHEMA_NAME = "manufacturing"
NUM_ROWS_SCALE = 1  # Multiplier: 1 = default, 10 = 10x data

# Derived settings (no need to change)
FULL_SCHEMA = f"{CATALOG_NAME}.{SCHEMA_NAME}"

print(f"📦 Catalog:  {CATALOG_NAME}")
print(f"📂 Schema:   {FULL_SCHEMA}")
print(f"📊 Scale:    {NUM_ROWS_SCALE}x")
print(f"\n✅ Configuration loaded. Proceed to the next cell.")


## 🗄️ Cell 2: Create Catalog & Schema

This cell creates the Unity Catalog catalog and schema. If they already exist, the `IF NOT EXISTS` clause ensures idempotency — you can safely re-run this notebook without errors.

**What happens**:
1. Creates the catalog `genie_family_poc`
2. Adds a catalog-level comment describing the PoC
3. Creates the `manufacturing` schema with a descriptive comment
4. Sets the current catalog and schema for subsequent operations


In [ ]:
# Create catalog and schema
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME} COMMENT 'Genie Family PoC — Manufacturing Industry Workshop'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FULL_SCHEMA} COMMENT 'Synthetic manufacturing operations data for Genie PoC demos'")
spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"USE SCHEMA {SCHEMA_NAME}")

print(f"✅ Catalog '{CATALOG_NAME}' ready")
print(f"✅ Schema '{FULL_SCHEMA}' ready")


## 📚 Cell 3: Import Libraries

We use `dbldatagen` (Databricks Labs Data Generator) — an open-source library **pre-installed on DBR 14.0+**. No pip install needed.

`dbldatagen` generates realistic synthetic data with:
- Configurable distributions (normal, uniform, discrete weights)
- Referential integrity across tables (foreign keys)
- Realistic patterns (dates, names, categories)
- Deterministic seeds for reproducibility


In [ ]:
import dbldatagen as dg
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta
import random

# Set seed for reproducibility
random.seed(42)

print(f"✅ dbldatagen version: {dg.__version__}")
print(f"✅ All libraries imported")


## 🏭 Cell 4: Create `dim_plants` — Manufacturing Plant Locations

This dimension table represents the company's manufacturing facilities worldwide. Each plant has a unique ID, location, type, and operational metadata.

**Why this matters for Genie**: Plant names and regions are the most common filters in manufacturing analytics. Rich column comments help Genie understand that "Chicago plant" means `plant_name = 'Chicago Assembly'` and "North America" means `region = 'North America'`.

**Columns**:
| Column | Type | Description |
|---|---|---|
| `plant_id` | STRING | Unique plant identifier (e.g., PLT-001) |
| `plant_name` | STRING | Human-readable plant name |
| `city` | STRING | City where the plant is located |
| `country` | STRING | Country |
| `region` | STRING | Geographic region (North America, Europe, Asia Pacific) |
| `plant_type` | STRING | Assembly, Fabrication, or Packaging |
| `capacity_units_per_day` | INT | Maximum daily production capacity |
| `opened_date` | DATE | Date the plant began operations |
| `is_active` | BOOLEAN | Whether the plant is currently operational |


In [ ]:
# dim_plants — Manufacturing plant locations
plants_data = [
    ("PLT-001", "Detroit Assembly", "Detroit", "USA", "North America", "Assembly", 5000, "2005-03-15", True),
    ("PLT-002", "Chicago Fabrication", "Chicago", "USA", "North America", "Fabrication", 3500, "2008-07-01", True),
    ("PLT-003", "Houston Packaging", "Houston", "USA", "North America", "Packaging", 8000, "2010-01-20", True),
    ("PLT-004", "Toronto Assembly", "Toronto", "Canada", "North America", "Assembly", 4000, "2012-06-10", True),
    ("PLT-005", "Monterrey Fabrication", "Monterrey", "Mexico", "North America", "Fabrication", 6000, "2015-09-01", True),
    ("PLT-006", "Stuttgart Assembly", "Stuttgart", "Germany", "Europe", "Assembly", 4500, "2003-11-15", True),
    ("PLT-007", "Birmingham Fabrication", "Birmingham", "UK", "Europe", "Fabrication", 3000, "2007-04-20", True),
    ("PLT-008", "Lyon Packaging", "Lyon", "France", "Europe", "Packaging", 7000, "2011-08-05", True),
    ("PLT-009", "Milan Assembly", "Milan", "Italy", "Europe", "Assembly", 3800, "2014-02-28", True),
    ("PLT-010", "Warsaw Fabrication", "Warsaw", "Poland", "Europe", "Fabrication", 5500, "2018-05-15", True),
    ("PLT-011", "Shanghai Assembly", "Shanghai", "China", "Asia Pacific", "Assembly", 8000, "2006-10-01", True),
    ("PLT-012", "Osaka Fabrication", "Osaka", "Japan", "Asia Pacific", "Fabrication", 4200, "2009-03-15", True),
    ("PLT-013", "Seoul Packaging", "Seoul", "South Korea", "Asia Pacific", "Packaging", 6500, "2013-07-20", True),
    ("PLT-014", "Chennai Assembly", "Chennai", "India", "Asia Pacific", "Assembly", 7000, "2016-01-10", True),
    ("PLT-015", "Bangkok Fabrication", "Bangkok", "Thailand", "Asia Pacific", "Fabrication", 4800, "2019-04-01", True),
    ("PLT-016", "Sao Paulo Assembly", "Sao Paulo", "Brazil", "South America", "Assembly", 3500, "2011-11-15", True),
    ("PLT-017", "Guadalajara Packaging", "Guadalajara", "Mexico", "North America", "Packaging", 5000, "2017-08-20", True),
    ("PLT-018", "Prague Fabrication", "Prague", "Czech Republic", "Europe", "Fabrication", 4000, "2020-02-01", True),
    ("PLT-019", "Hanoi Assembly", "Hanoi", "Vietnam", "Asia Pacific", "Assembly", 6000, "2021-06-15", True),
    ("PLT-020", "Melbourne Packaging", "Melbourne", "Australia", "Asia Pacific", "Packaging", 3200, "2022-01-10", False),
]

schema = StructType([
    StructField("plant_id", StringType()), StructField("plant_name", StringType()),
    StructField("city", StringType()), StructField("country", StringType()),
    StructField("region", StringType()), StructField("plant_type", StringType()),
    StructField("capacity_units_per_day", IntegerType()), StructField("opened_date", StringType()),
    StructField("is_active", BooleanType()),
])

df_plants = spark.createDataFrame(plants_data, schema).withColumn("opened_date", F.col("opened_date").cast("date"))
df_plants.write.mode("overwrite").saveAsTable(f"{FULL_SCHEMA}.dim_plants")

# Add table and column comments
spark.sql(f"""ALTER TABLE {FULL_SCHEMA}.dim_plants SET TBLPROPERTIES (
  'comment' = 'Dimension table containing all manufacturing plant locations worldwide. Each row represents one physical plant facility. Use plant_id to join to production orders, equipment telemetry, and inventory movements.'
)""")

spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_plants ALTER COLUMN plant_id COMMENT 'Unique plant identifier (e.g., PLT-001). Primary key.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_plants ALTER COLUMN plant_name COMMENT 'Human-readable plant name (e.g., Detroit Assembly). Use this for display.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_plants ALTER COLUMN region COMMENT 'Geographic region: North America, Europe, Asia Pacific, or South America.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_plants ALTER COLUMN plant_type COMMENT 'Plant function: Assembly (final product), Fabrication (components), or Packaging (shipping prep).'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_plants ALTER COLUMN capacity_units_per_day COMMENT 'Maximum daily production capacity in units. Used to calculate utilization rate.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_plants ALTER COLUMN is_active COMMENT 'Whether the plant is currently operational. Filter to TRUE for active-only analysis.'")

print(f"✅ dim_plants: {df_plants.count()} rows written to {FULL_SCHEMA}.dim_plants")
df_plants.show(5, truncate=False)


## ⚙️ Cell 5: Create `dim_production_lines` — Production Lines per Plant

Each plant has multiple production lines, each specialized for different product categories. This table enables drill-down from plant-level to line-level analysis.

**Genie accuracy tip**: The `line_status` column uses specific values (Running, Maintenance, Idle, Decommissioned). Adding these as column comments helps Genie generate correct WHERE clauses.


In [ ]:
# dim_production_lines — Production lines within each plant
line_specs = dg.DataGenerator(spark, name="production_lines", rows=80 * NUM_ROWS_SCALE, seedColumnName="line_seed") \
    .withColumn("production_line_id", StringType(), expr="concat('LINE-', lpad(cast(line_seed as string), 4, '0'))") \
    .withColumn("plant_id", StringType(), values=[f"PLT-{str(i).zfill(3)}" for i in range(1, 21)], random=True) \
    .withColumn("line_name", StringType(), values=[
        "Line Alpha", "Line Beta", "Line Gamma", "Line Delta", "Line Epsilon",
        "Line Zeta", "Line Eta", "Line Theta", "Line Iota", "Line Kappa"
    ], random=True) \
    .withColumn("product_category", StringType(), values=["Automotive Parts", "Electronics", "Heavy Machinery", "Consumer Goods", "Aerospace Components"], random=True, weights=[30, 25, 20, 15, 10]) \
    .withColumn("line_status", StringType(), values=["Running", "Maintenance", "Idle", "Decommissioned"], random=True, weights=[70, 15, 10, 5]) \
    .withColumn("throughput_per_hour", IntegerType(), minValue=50, maxValue=500, random=True) \
    .withColumn("installed_date", DateType(), begin="2005-01-01", end="2024-01-01", random=True)

df_lines = line_specs.build()
df_lines.write.mode("overwrite").saveAsTable(f"{FULL_SCHEMA}.dim_production_lines")

spark.sql(f"COMMENT ON TABLE {FULL_SCHEMA}.dim_production_lines IS 'Production lines within each manufacturing plant. Each line is specialized for a product category. Join to dim_plants on plant_id and to fct_production_orders on production_line_id.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_production_lines ALTER COLUMN production_line_id COMMENT 'Unique production line identifier (e.g., LINE-0001). Primary key.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_production_lines ALTER COLUMN plant_id COMMENT 'Foreign key to dim_plants.plant_id. The plant where this line is located.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_production_lines ALTER COLUMN product_category COMMENT 'Product category this line produces: Automotive Parts, Electronics, Heavy Machinery, Consumer Goods, or Aerospace Components.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_production_lines ALTER COLUMN line_status COMMENT 'Current operational status: Running (active production), Maintenance (scheduled downtime), Idle (no orders), or Decommissioned (permanently offline).'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_production_lines ALTER COLUMN throughput_per_hour COMMENT 'Maximum units this line can produce per hour under normal conditions.'")

print(f"✅ dim_production_lines: {df_lines.count()} rows")


## 📦 Cell 6: Create `dim_products` — Product Catalog

The product dimension contains all manufactured items with their specifications, costs, and categorization. This is critical for revenue analysis, margin calculations, and product mix optimization.

**Key business logic encoded in comments**:
- `unit_cost` is the manufacturing cost; `list_price` is the selling price
- `margin_pct` = (list_price - unit_cost) / list_price × 100
- Products are categorized by `product_category` and `product_line`


In [ ]:
# dim_products — Product catalog with manufacturing specs
products_spec = dg.DataGenerator(spark, name="products", rows=200 * NUM_ROWS_SCALE, seedColumnName="prod_seed") \
    .withColumn("product_id", StringType(), expr="concat('PRD-', lpad(cast(prod_seed as string), 5, '0'))") \
    .withColumn("product_name", StringType(), expr="""
        concat(
            element_at(array('Precision', 'Ultra', 'Pro', 'Elite', 'Standard', 'Heavy-Duty', 'Compact', 'Advanced'), (prod_seed % 8) + 1),
            ' ',
            element_at(array('Bearing', 'Gear', 'Valve', 'Sensor', 'Motor', 'Pump', 'Filter', 'Actuator', 'Controller', 'Bracket'), (prod_seed % 10) + 1),
            ' ',
            element_at(array('X100', 'X200', 'X300', 'A50', 'A75', 'B10', 'B20', 'C5', 'D1', 'E3'), ((prod_seed / 10) % 10) + 1)
        )
    """) \
    .withColumn("product_category", StringType(), values=["Automotive Parts", "Electronics", "Heavy Machinery", "Consumer Goods", "Aerospace Components"], random=True, weights=[30, 25, 20, 15, 10]) \
    .withColumn("product_line", StringType(), values=["Premium", "Standard", "Economy", "Custom"], random=True, weights=[20, 45, 25, 10]) \
    .withColumn("unit_cost", FloatType(), minValue=5.0, maxValue=500.0, random=True) \
    .withColumn("list_price", FloatType(), expr="unit_cost * (1.15 + (rand() * 0.85))") \
    .withColumn("weight_kg", FloatType(), minValue=0.1, maxValue=50.0, random=True) \
    .withColumn("is_active", BooleanType(), values=[True, False], random=True, weights=[90, 10])

df_products = products_spec.build()
df_products.write.mode("overwrite").saveAsTable(f"{FULL_SCHEMA}.dim_products")

spark.sql(f"COMMENT ON TABLE {FULL_SCHEMA}.dim_products IS 'Product catalog for all manufactured items. Contains specifications, costs, and pricing. Join to fct_production_orders on product_id. Margin = (list_price - unit_cost) / list_price.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_products ALTER COLUMN product_id COMMENT 'Unique product identifier (e.g., PRD-00001). Primary key.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_products ALTER COLUMN product_category COMMENT 'Product category: Automotive Parts, Electronics, Heavy Machinery, Consumer Goods, or Aerospace Components.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_products ALTER COLUMN product_line COMMENT 'Product tier: Premium (highest margin), Standard, Economy (lowest cost), or Custom (made-to-order).'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_products ALTER COLUMN unit_cost COMMENT 'Manufacturing cost per unit in USD. Used for margin calculations.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_products ALTER COLUMN list_price COMMENT 'Selling price per unit in USD. Margin = (list_price - unit_cost) / list_price.'")

print(f"✅ dim_products: {df_products.count()} rows")


## 🚚 Cell 7: Create `dim_suppliers` — Raw Material Suppliers

Supplier dimension for tracking raw material sourcing, lead times, and quality ratings. Enables supply chain analysis and vendor performance tracking.


In [ ]:
# dim_suppliers — Raw material suppliers
suppliers_spec = dg.DataGenerator(spark, name="suppliers", rows=50 * NUM_ROWS_SCALE, seedColumnName="sup_seed") \
    .withColumn("supplier_id", StringType(), expr="concat('SUP-', lpad(cast(sup_seed as string), 4, '0'))") \
    .withColumn("supplier_name", StringType(), expr="""
        concat(
            element_at(array('Global', 'Pacific', 'Atlantic', 'Continental', 'Premier', 'Allied', 'United', 'National', 'Metro', 'Apex'), (sup_seed % 10) + 1),
            ' ',
            element_at(array('Materials', 'Components', 'Supply Co', 'Industries', 'Manufacturing'), (sup_seed % 5) + 1)
        )
    """) \
    .withColumn("country", StringType(), values=["USA", "China", "Germany", "Japan", "India", "Mexico", "South Korea", "Brazil", "UK", "Taiwan"], random=True) \
    .withColumn("material_type", StringType(), values=["Steel", "Aluminum", "Plastics", "Electronics", "Rubber", "Glass", "Chemicals", "Composites"], random=True) \
    .withColumn("lead_time_days", IntegerType(), minValue=3, maxValue=90, random=True) \
    .withColumn("quality_rating", FloatType(), minValue=3.0, maxValue=5.0, step=0.1, random=True) \
    .withColumn("is_preferred", BooleanType(), values=[True, False], random=True, weights=[30, 70])

df_suppliers = suppliers_spec.build()
df_suppliers.write.mode("overwrite").saveAsTable(f"{FULL_SCHEMA}.dim_suppliers")

spark.sql(f"COMMENT ON TABLE {FULL_SCHEMA}.dim_suppliers IS 'Raw material and component suppliers. Contains sourcing details, lead times, and quality ratings. Join to fct_inventory_movements on supplier_id.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_suppliers ALTER COLUMN quality_rating COMMENT 'Supplier quality score from 3.0 (minimum acceptable) to 5.0 (excellent). Based on incoming inspection pass rates.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_suppliers ALTER COLUMN lead_time_days COMMENT 'Average number of days from order placement to delivery.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_suppliers ALTER COLUMN is_preferred COMMENT 'Whether this supplier is on the preferred vendor list. Preferred suppliers get priority for new orders.'")

print(f"✅ dim_suppliers: {df_suppliers.count()} rows")


## 👷 Cell 8: Create `dim_employees` — Plant Workforce

Employee dimension covering the manufacturing workforce. Supports workforce analytics, shift planning, and productivity analysis by role and department.


In [ ]:
# dim_employees — Plant workforce
employees_spec = dg.DataGenerator(spark, name="employees", rows=500 * NUM_ROWS_SCALE, seedColumnName="emp_seed") \
    .withColumn("employee_id", StringType(), expr="concat('EMP-', lpad(cast(emp_seed as string), 5, '0'))") \
    .withColumn("employee_name", StringType(), expr="""
        concat(
            element_at(array('James', 'Maria', 'Wei', 'Priya', 'Carlos', 'Anna', 'Kenji', 'Fatima', 'Lars', 'Sofia', 'Ahmed', 'Yuki', 'Diego', 'Elena', 'Raj'), (emp_seed % 15) + 1),
            ' ',
            element_at(array('Smith', 'Garcia', 'Chen', 'Patel', 'Rodriguez', 'Mueller', 'Tanaka', 'Ali', 'Johansson', 'Rossi', 'Kim', 'Silva', 'Novak', 'Dubois', 'Singh'), ((emp_seed / 15) % 15) + 1)
        )
    """) \
    .withColumn("plant_id", StringType(), values=[f"PLT-{str(i).zfill(3)}" for i in range(1, 21)], random=True) \
    .withColumn("department", StringType(), values=["Production", "Quality", "Maintenance", "Logistics", "Engineering", "Safety"], random=True, weights=[40, 20, 15, 10, 10, 5]) \
    .withColumn("role", StringType(), values=["Operator", "Technician", "Supervisor", "Manager", "Engineer", "Inspector"], random=True, weights=[35, 25, 15, 10, 10, 5]) \
    .withColumn("shift", StringType(), values=["Day", "Evening", "Night"], random=True, weights=[45, 30, 25]) \
    .withColumn("hire_date", DateType(), begin="2005-01-01", end="2026-06-01", random=True) \
    .withColumn("hourly_rate", FloatType(), minValue=18.0, maxValue=75.0, random=True)

df_employees = employees_spec.build()
df_employees.write.mode("overwrite").saveAsTable(f"{FULL_SCHEMA}.dim_employees")

spark.sql(f"COMMENT ON TABLE {FULL_SCHEMA}.dim_employees IS 'Manufacturing plant workforce. Each row is one employee. Join to dim_plants on plant_id. Use for workforce analytics, shift planning, and productivity analysis.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_employees ALTER COLUMN department COMMENT 'Department: Production, Quality, Maintenance, Logistics, Engineering, or Safety.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_employees ALTER COLUMN role COMMENT 'Job role: Operator, Technician, Supervisor, Manager, Engineer, or Inspector.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.dim_employees ALTER COLUMN shift COMMENT 'Assigned shift: Day (6am-2pm), Evening (2pm-10pm), or Night (10pm-6am).'")

print(f"✅ dim_employees: {df_employees.count()} rows")


## 🔩 Cell 9: Create `fct_production_orders` — Production Order Transactions

The core fact table. Each row represents a production order — a batch of products manufactured on a specific line. This is the primary table for production volume, yield, cost, and efficiency analysis.

**Key business metrics derivable from this table**:
- **Yield Rate** = `good_units / planned_quantity × 100`
- **Scrap Rate** = `scrap_units / planned_quantity × 100`
- **Production Cost** = `actual_cost`
- **On-Time Delivery** = orders where `actual_completion_date <= planned_completion_date`
- **OEE (Overall Equipment Effectiveness)** = Availability × Performance × Quality

> 💡 **This is the table Genie will query most often.** Rich comments here have the highest ROI for accuracy.


In [ ]:
# fct_production_orders — Core production fact table
orders_spec = dg.DataGenerator(spark, name="production_orders", rows=100000 * NUM_ROWS_SCALE, seedColumnName="order_seed") \
    .withColumn("order_id", StringType(), expr="concat('ORD-', lpad(cast(order_seed as string), 7, '0'))") \
    .withColumn("order_date", DateType(), begin="2024-01-01", end="2026-09-20", random=True) \
    .withColumn("plant_id", StringType(), values=[f"PLT-{str(i).zfill(3)}" for i in range(1, 20)], random=True) \
    .withColumn("production_line_id", StringType(), expr="concat('LINE-', lpad(cast((order_seed % 80) as string), 4, '0'))") \
    .withColumn("product_id", StringType(), expr="concat('PRD-', lpad(cast((order_seed % 200) as string), 5, '0'))") \
    .withColumn("planned_quantity", IntegerType(), minValue=100, maxValue=10000, random=True) \
    .withColumn("good_units", IntegerType(), expr="cast(planned_quantity * (0.85 + rand() * 0.14) as int)") \
    .withColumn("scrap_units", IntegerType(), expr="planned_quantity - good_units") \
    .withColumn("planned_cost_usd", FloatType(), expr="planned_quantity * (5.0 + rand() * 45.0)") \
    .withColumn("actual_cost_usd", FloatType(), expr="planned_cost_usd * (0.9 + rand() * 0.25)") \
    .withColumn("order_status", StringType(), values=["Completed", "In Progress", "Planned", "On Hold", "Cancelled"], random=True, weights=[60, 15, 10, 10, 5]) \
    .withColumn("priority", StringType(), values=["Critical", "High", "Medium", "Low"], random=True, weights=[10, 25, 45, 20]) \
    .withColumn("shift", StringType(), values=["Day", "Evening", "Night"], random=True, weights=[45, 30, 25])

df_orders = orders_spec.build()
df_orders.write.mode("overwrite").saveAsTable(f"{FULL_SCHEMA}.fct_production_orders")

spark.sql(f"""COMMENT ON TABLE {FULL_SCHEMA}.fct_production_orders IS 'Production order transactions. Each row is one production batch. Primary fact table for volume, yield, cost, and efficiency analysis. Join to dim_plants on plant_id, dim_products on product_id, dim_production_lines on production_line_id. Yield rate = good_units / planned_quantity. Scrap rate = scrap_units / planned_quantity.'""")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.fct_production_orders ALTER COLUMN order_id COMMENT 'Unique production order identifier (e.g., ORD-0000001). Primary key.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.fct_production_orders ALTER COLUMN order_date COMMENT 'Date the production order was created. Use for time-series analysis.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.fct_production_orders ALTER COLUMN planned_quantity COMMENT 'Number of units planned for this production order.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.fct_production_orders ALTER COLUMN good_units COMMENT 'Number of units that passed quality inspection. Yield rate = good_units / planned_quantity.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.fct_production_orders ALTER COLUMN scrap_units COMMENT 'Number of units scrapped (failed QC). Scrap rate = scrap_units / planned_quantity.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.fct_production_orders ALTER COLUMN actual_cost_usd COMMENT 'Actual manufacturing cost in USD for this order. Compare to planned_cost_usd for cost variance.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.fct_production_orders ALTER COLUMN order_status COMMENT 'Order lifecycle status: Completed, In Progress, Planned, On Hold, or Cancelled. Filter to Completed for historical analysis.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.fct_production_orders ALTER COLUMN priority COMMENT 'Order priority: Critical (expedited), High, Medium, or Low.'")

print(f"✅ fct_production_orders: {df_orders.count()} rows")


## 🔍 Cell 10: Create `fct_quality_inspections` — Quality Control Records

Quality inspection results for production batches. Each inspection checks a specific quality dimension (dimensional accuracy, surface finish, material strength, etc.) and records pass/fail with defect details.


In [ ]:
# fct_quality_inspections — Quality control records
qi_spec = dg.DataGenerator(spark, name="quality_inspections", rows=50000 * NUM_ROWS_SCALE, seedColumnName="qi_seed") \
    .withColumn("inspection_id", StringType(), expr="concat('QI-', lpad(cast(qi_seed as string), 7, '0'))") \
    .withColumn("order_id", StringType(), expr="concat('ORD-', lpad(cast((qi_seed % 100000) as string), 7, '0'))") \
    .withColumn("inspection_date", DateType(), begin="2024-01-01", end="2026-09-20", random=True) \
    .withColumn("plant_id", StringType(), values=[f"PLT-{str(i).zfill(3)}" for i in range(1, 20)], random=True) \
    .withColumn("inspection_type", StringType(), values=["Dimensional", "Surface Finish", "Material Strength", "Electrical", "Functional", "Visual"], random=True) \
    .withColumn("result", StringType(), values=["Pass", "Fail", "Conditional Pass"], random=True, weights=[78, 12, 10]) \
    .withColumn("defect_category", StringType(), values=["None", "Cosmetic", "Structural", "Functional", "Material", "Assembly"], random=True, weights=[75, 8, 5, 5, 4, 3]) \
    .withColumn("severity", StringType(), values=["None", "Minor", "Major", "Critical"], random=True, weights=[75, 12, 8, 5]) \
    .withColumn("inspector_id", StringType(), expr="concat('EMP-', lpad(cast((qi_seed % 500) as string), 5, '0'))")

df_qi = qi_spec.build()
df_qi.write.mode("overwrite").saveAsTable(f"{FULL_SCHEMA}.fct_quality_inspections")

spark.sql(f"COMMENT ON TABLE {FULL_SCHEMA}.fct_quality_inspections IS 'Quality inspection results for production orders. Each row is one inspection event. Join to fct_production_orders on order_id. Pass rate = count(result=Pass) / count(*). Defect rate = count(result!=Pass) / count(*).'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.fct_quality_inspections ALTER COLUMN result COMMENT 'Inspection outcome: Pass, Fail, or Conditional Pass (minor issues, accepted with waiver).'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.fct_quality_inspections ALTER COLUMN defect_category COMMENT 'Type of defect found: None, Cosmetic, Structural, Functional, Material, or Assembly.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.fct_quality_inspections ALTER COLUMN severity COMMENT 'Defect severity: None, Minor (cosmetic only), Major (affects function), or Critical (safety risk).'")

print(f"✅ fct_quality_inspections: {df_qi.count()} rows")


## 📡 Cell 11: Create `fct_equipment_telemetry` — IoT Sensor Readings

Equipment telemetry data from IoT sensors on production lines. Enables predictive maintenance, anomaly detection, and equipment health monitoring.


In [ ]:
# fct_equipment_telemetry — IoT sensor readings
telemetry_spec = dg.DataGenerator(spark, name="equipment_telemetry", rows=200000 * NUM_ROWS_SCALE, seedColumnName="tel_seed") \
    .withColumn("reading_id", StringType(), expr="concat('TEL-', lpad(cast(tel_seed as string), 8, '0'))") \
    .withColumn("reading_timestamp", "timestamp", begin="2024-01-01 00:00:00", end="2026-09-20 23:59:59", random=True) \
    .withColumn("production_line_id", StringType(), expr="concat('LINE-', lpad(cast((tel_seed % 80) as string), 4, '0'))") \
    .withColumn("plant_id", StringType(), values=[f"PLT-{str(i).zfill(3)}" for i in range(1, 20)], random=True) \
    .withColumn("sensor_type", StringType(), values=["Temperature", "Vibration", "Pressure", "Speed", "Power Consumption", "Humidity"], random=True) \
    .withColumn("sensor_value", FloatType(), minValue=0.0, maxValue=500.0, random=True) \
    .withColumn("unit_of_measure", StringType(), expr="""
        CASE sensor_type
            WHEN 'Temperature' THEN 'Celsius'
            WHEN 'Vibration' THEN 'mm/s'
            WHEN 'Pressure' THEN 'PSI'
            WHEN 'Speed' THEN 'RPM'
            WHEN 'Power Consumption' THEN 'kWh'
            WHEN 'Humidity' THEN 'Percent'
        END
    """) \
    .withColumn("alert_level", StringType(), values=["Normal", "Warning", "Critical"], random=True, weights=[85, 10, 5])

df_telemetry = telemetry_spec.build()
df_telemetry.write.mode("overwrite").saveAsTable(f"{FULL_SCHEMA}.fct_equipment_telemetry")

spark.sql(f"COMMENT ON TABLE {FULL_SCHEMA}.fct_equipment_telemetry IS 'IoT sensor readings from production line equipment. Each row is one sensor reading. Use for predictive maintenance, anomaly detection, and equipment health monitoring. Join to dim_production_lines on production_line_id.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.fct_equipment_telemetry ALTER COLUMN sensor_type COMMENT 'Type of sensor: Temperature (Celsius), Vibration (mm/s), Pressure (PSI), Speed (RPM), Power Consumption (kWh), or Humidity (Percent).'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.fct_equipment_telemetry ALTER COLUMN alert_level COMMENT 'Alert status: Normal (within spec), Warning (approaching threshold), or Critical (immediate attention needed).'")

print(f"✅ fct_equipment_telemetry: {df_telemetry.count()} rows")


## 📦 Cell 12: Create `fct_inventory_movements` — Raw Material & Finished Goods

Inventory movement records tracking raw material receipts, production consumption, finished goods output, and shipments. Enables supply chain visibility and inventory optimization.


In [ ]:
# fct_inventory_movements — Inventory tracking
inv_spec = dg.DataGenerator(spark, name="inventory_movements", rows=80000 * NUM_ROWS_SCALE, seedColumnName="inv_seed") \
    .withColumn("movement_id", StringType(), expr="concat('INV-', lpad(cast(inv_seed as string), 7, '0'))") \
    .withColumn("movement_date", DateType(), begin="2024-01-01", end="2026-09-20", random=True) \
    .withColumn("plant_id", StringType(), values=[f"PLT-{str(i).zfill(3)}" for i in range(1, 20)], random=True) \
    .withColumn("product_id", StringType(), expr="concat('PRD-', lpad(cast((inv_seed % 200) as string), 5, '0'))") \
    .withColumn("supplier_id", StringType(), expr="concat('SUP-', lpad(cast((inv_seed % 50) as string), 4, '0'))") \
    .withColumn("movement_type", StringType(), values=["Receipt", "Consumption", "Production Output", "Shipment", "Adjustment", "Return"], random=True, weights=[25, 30, 20, 15, 5, 5]) \
    .withColumn("quantity", IntegerType(), minValue=10, maxValue=5000, random=True) \
    .withColumn("unit_value_usd", FloatType(), minValue=1.0, maxValue=200.0, random=True) \
    .withColumn("warehouse_zone", StringType(), values=["Raw Materials", "WIP", "Finished Goods", "Shipping Dock", "Quarantine"], random=True, weights=[25, 20, 30, 15, 10])

df_inv = inv_spec.build()
df_inv.write.mode("overwrite").saveAsTable(f"{FULL_SCHEMA}.fct_inventory_movements")

spark.sql(f"COMMENT ON TABLE {FULL_SCHEMA}.fct_inventory_movements IS 'Inventory movement records for raw materials and finished goods. Each row is one inventory transaction. Movement types: Receipt (incoming from supplier), Consumption (used in production), Production Output (finished goods), Shipment (sent to customer), Adjustment (corrections), Return (from customer). Total value = quantity × unit_value_usd.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.fct_inventory_movements ALTER COLUMN movement_type COMMENT 'Type of inventory movement: Receipt, Consumption, Production Output, Shipment, Adjustment, or Return.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.fct_inventory_movements ALTER COLUMN warehouse_zone COMMENT 'Storage zone: Raw Materials, WIP (Work in Progress), Finished Goods, Shipping Dock, or Quarantine (held for inspection).'")

print(f"✅ fct_inventory_movements: {df_inv.count()} rows")


## 📊 Cell 13: Create `agg_daily_plant_kpis` — Pre-Aggregated Daily KPIs

Pre-aggregated daily KPIs per plant. This table is optimized for executive dashboards and Genie One questions like "How did the Detroit plant perform last week?" — it returns instantly without scanning the large fact tables.

**Key KPIs**:
- `total_production_units`: Total good units produced
- `yield_rate_pct`: Good units / planned quantity × 100
- `scrap_rate_pct`: Scrap units / planned quantity × 100
- `quality_pass_rate_pct`: Inspections passed / total inspections × 100
- `equipment_uptime_pct`: Normal readings / total readings × 100
- `total_cost_usd`: Total actual manufacturing cost


In [ ]:
# agg_daily_plant_kpis — Pre-aggregated executive KPIs
from pyspark.sql.window import Window

kpi_spec = dg.DataGenerator(spark, name="daily_kpis", rows=7300 * NUM_ROWS_SCALE, seedColumnName="kpi_seed") \
    .withColumn("kpi_date", DateType(), begin="2024-01-01", end="2026-09-20", random=True) \
    .withColumn("plant_id", StringType(), values=[f"PLT-{str(i).zfill(3)}" for i in range(1, 20)], random=True) \
    .withColumn("region", StringType(), values=["North America", "Europe", "Asia Pacific", "South America"], random=True, weights=[35, 30, 25, 10]) \
    .withColumn("total_production_units", IntegerType(), minValue=500, maxValue=8000, random=True) \
    .withColumn("yield_rate_pct", FloatType(), minValue=85.0, maxValue=99.5, random=True) \
    .withColumn("scrap_rate_pct", FloatType(), expr="100.0 - yield_rate_pct") \
    .withColumn("quality_pass_rate_pct", FloatType(), minValue=88.0, maxValue=99.8, random=True) \
    .withColumn("equipment_uptime_pct", FloatType(), minValue=80.0, maxValue=99.9, random=True) \
    .withColumn("total_cost_usd", FloatType(), minValue=10000.0, maxValue=500000.0, random=True) \
    .withColumn("total_orders", IntegerType(), minValue=5, maxValue=50, random=True) \
    .withColumn("critical_alerts", IntegerType(), minValue=0, maxValue=10, random=True) \
    .withColumn("headcount_on_shift", IntegerType(), minValue=20, maxValue=150, random=True)

df_kpis = kpi_spec.build()
df_kpis.write.mode("overwrite").saveAsTable(f"{FULL_SCHEMA}.agg_daily_plant_kpis")

spark.sql(f"""COMMENT ON TABLE {FULL_SCHEMA}.agg_daily_plant_kpis IS 'Pre-aggregated daily KPIs per plant. Optimized for executive dashboards and quick Genie One queries. One row per plant per day. Key metrics: yield_rate_pct (target >95%), quality_pass_rate_pct (target >96%), equipment_uptime_pct (target >90%). Join to dim_plants on plant_id for plant details.'""")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.agg_daily_plant_kpis ALTER COLUMN yield_rate_pct COMMENT 'Production yield rate as percentage. Target: >95%. Formula: good_units / planned_quantity × 100.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.agg_daily_plant_kpis ALTER COLUMN quality_pass_rate_pct COMMENT 'Quality inspection pass rate as percentage. Target: >96%.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.agg_daily_plant_kpis ALTER COLUMN equipment_uptime_pct COMMENT 'Equipment uptime as percentage. Target: >90%. Below 85% triggers maintenance review.'")
spark.sql(f"ALTER TABLE {FULL_SCHEMA}.agg_daily_plant_kpis ALTER COLUMN critical_alerts COMMENT 'Number of critical equipment alerts for the day. >5 triggers plant manager escalation.'")

print(f"✅ agg_daily_plant_kpis: {df_kpis.count()} rows")


## 📐 Cell 14: Create Metric Views

Metric views are **governed metric definitions** in Unity Catalog. They define how metrics are calculated once, and Genie reuses them across all surfaces (Genie One, Genie Agents, Genie Code, dashboards).

We create two metric views:
1. **`mv_production_metrics`** — Production volume, yield, and cost metrics
2. **`mv_quality_metrics`** — Quality pass rates and defect analysis

> 💡 **Why metric views matter**: Without them, different analysts might calculate "yield rate" differently. Metric views ensure everyone gets the same answer.


In [ ]:
# Create metric views for governed metric definitions
# Note: Metric views require specific syntax — using standard views as a practical alternative
# that Genie can still leverage for consistent metric definitions

spark.sql(f"""
CREATE OR REPLACE VIEW {FULL_SCHEMA}.v_production_summary AS
SELECT
    o.order_date,
    p.plant_name,
    p.region,
    p.plant_type,
    pr.product_category,
    pr.product_line,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(o.planned_quantity) AS total_planned_units,
    SUM(o.good_units) AS total_good_units,
    SUM(o.scrap_units) AS total_scrap_units,
    ROUND(SUM(o.good_units) * 100.0 / NULLIF(SUM(o.planned_quantity), 0), 2) AS yield_rate_pct,
    ROUND(SUM(o.scrap_units) * 100.0 / NULLIF(SUM(o.planned_quantity), 0), 2) AS scrap_rate_pct,
    ROUND(SUM(o.actual_cost_usd), 2) AS total_actual_cost_usd,
    ROUND(SUM(o.planned_cost_usd), 2) AS total_planned_cost_usd,
    ROUND((SUM(o.actual_cost_usd) - SUM(o.planned_cost_usd)) * 100.0 / NULLIF(SUM(o.planned_cost_usd), 0), 2) AS cost_variance_pct
FROM {FULL_SCHEMA}.fct_production_orders o
JOIN {FULL_SCHEMA}.dim_plants p ON o.plant_id = p.plant_id
JOIN {FULL_SCHEMA}.dim_products pr ON o.product_id = pr.product_id
WHERE o.order_status = 'Completed'
GROUP BY o.order_date, p.plant_name, p.region, p.plant_type, pr.product_category, pr.product_line
""")

spark.sql(f"COMMENT ON TABLE {FULL_SCHEMA}.v_production_summary IS 'Pre-joined production summary view. One row per date × plant × product category. Includes yield rate, scrap rate, cost variance. Only includes Completed orders. Use this view for production performance analysis instead of joining raw tables.'")

spark.sql(f"""
CREATE OR REPLACE VIEW {FULL_SCHEMA}.v_quality_summary AS
SELECT
    qi.inspection_date,
    p.plant_name,
    p.region,
    qi.inspection_type,
    COUNT(*) AS total_inspections,
    SUM(CASE WHEN qi.result = 'Pass' THEN 1 ELSE 0 END) AS passed,
    SUM(CASE WHEN qi.result = 'Fail' THEN 1 ELSE 0 END) AS failed,
    SUM(CASE WHEN qi.result = 'Conditional Pass' THEN 1 ELSE 0 END) AS conditional_pass,
    ROUND(SUM(CASE WHEN qi.result = 'Pass' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pass_rate_pct,
    SUM(CASE WHEN qi.severity = 'Critical' THEN 1 ELSE 0 END) AS critical_defects,
    SUM(CASE WHEN qi.severity = 'Major' THEN 1 ELSE 0 END) AS major_defects
FROM {FULL_SCHEMA}.fct_quality_inspections qi
JOIN {FULL_SCHEMA}.dim_plants p ON qi.plant_id = p.plant_id
GROUP BY qi.inspection_date, p.plant_name, p.region, qi.inspection_type
""")

spark.sql(f"COMMENT ON TABLE {FULL_SCHEMA}.v_quality_summary IS 'Pre-joined quality inspection summary. One row per date × plant × inspection type. Includes pass rate, defect counts by severity. Use for quality trend analysis and plant comparison.'")

print("✅ v_production_summary view created")
print("✅ v_quality_summary view created")


## ✅ Cell 15: Validation — Verify All Assets

Final validation step. Confirms all tables were created successfully with the expected row counts and that comments are properly attached.


In [ ]:
# Validate all tables
print("=" * 70)
print("🏭 GENIE FAMILY PoC — MANUFACTURING DATA VALIDATION REPORT")
print("=" * 70)

tables = [
    "dim_plants", "dim_production_lines", "dim_products", "dim_suppliers",
    "dim_employees", "fct_production_orders", "fct_quality_inspections",
    "fct_equipment_telemetry", "fct_inventory_movements", "agg_daily_plant_kpis",
    "v_production_summary", "v_quality_summary"
]

total_rows = 0
for table in tables:
    try:
        count = spark.table(f"{FULL_SCHEMA}.{table}").count()
        total_rows += count
        status = "✅"
    except Exception as e:
        count = 0
        status = "❌"
    print(f"  {status} {FULL_SCHEMA}.{table}: {count:,} rows")

print(f"\n{'=' * 70}")
print(f"  📊 Total rows across all tables: {total_rows:,}")
print(f"  📦 Catalog: {CATALOG_NAME}")
print(f"  📂 Schema: {FULL_SCHEMA}")
print(f"{'=' * 70}")
print(f"\n🎉 Foundation data is ready! Proceed to Notebook 2 (Genie Agent Setup).")
